# Lesson 3.3: Human-in-the-loop

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [3]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.chat_models import init_chat_model

class EmailState(AgentState):
    email: str

gemini_lite = init_chat_model(
    model="models/gemini-3.5-flash-lite", 
    model_provider="google_genai"
)

# Update the agent creation cell:

agent = create_agent(
    model=gemini_lite,
    tools=[read_email, send_email],
    state_schema=EmailState,
    system_prompt="""You are an executive email assistant. 
    Always use the send_email tool to send email replies.
    If your send_email tool call is rejected or you receive feedback, you MUST incorporate the feedback and immediately call the send_email tool again with the updated body.""",
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [4]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [5]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'no '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          'Let '
                                                                          'me '
                                                                          'know '
                                                                          'what '
                                                                          'times '
                                                                          'work '
                    

In [6]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi John, no problem at all. Let me know what times work best for you to reschedule. Best, Seán'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Hi John, no problem at all. Let me know what times work best for you to reschedule. Best, Seán'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='792c362a757d5fbe5ec71785993755cb')]


In [7]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, no problem at all. Let me know what times work best for you to reschedule. Best, Seán


# 2.1 Approve

In [ ]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

# 2.2 Reject

In [9]:
from langgraph.types import Command

response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
)   

pprint(response)

# Safe inspection: checks whether the model re-attempted the tool or responded with text
if "__interrupt__" in response:
    print("\n⏸️ Interrupted on new tool call! Updated body:")
    print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])
else:
    print("\n💬 Agent finished with message:")
    ans = response['messages'][-1].content
    print(ans[0]['text'] if isinstance(ans, list) else ans)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, \n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem '
                                                                          'at '
                                                                          'all. '
                                                                          'Let '
                                                                          'me '
                                                                          'know '
                                                                          'what '
                                                                          'time '
                      

In [10]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, 

No problem at all. Let me know what time works best for you tomorrow and we can reschedule our meeting. 

Best, 
Seán


# 2.3: Edit

In [9]:
from langgraph.types import Command

response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='172f753f-2a6c-4b97-83d6-c077ffd95be6'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_321679': 'El4KXAERTTIP7CV2/NK2dF753BugcC5gY0OFv30ZBWVtxePKOR9OqyhPnAFhjONbHVWJmf04X+WAKkuNOibGQnuIwJ5dZA2zfmTJR9QXTn5blVOwBIdu8/o0oNtXYcq6'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0449c-801e-76d2-b001-49dbbcf411b2-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'call_321679', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_toke